<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T05-L01-serving-on-a-cpu/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T05-L01-serving-on-a-cpu/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T05-L01-serving-on-a-cpu/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T05-L01-serving-on-a-cpu/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T05-L01 · Serving a small model on a CPU: the KV cache, batching, and what each one costs

**You will build:** the two ways an inference engine can decode with a small transformer (recompute
everything each step, or keep a key-value cache), the exact accounting of the arithmetic and the
memory each one uses, and a static batcher whose padding mistakes you will measure rather than
be told about.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, no GPU, no download, well under a minute ·
**Prerequisites:** T00-L01-the-8gb-track, T03-L03-nanolm-cpu-slice.

The model in this lesson has random weights and cannot write. That is deliberate. Serving is
about computing the SAME function faster and in less memory, so every comparison below is between
two ways of computing one model, and it needs no trained weights and no language model.

By the end you will be able to:

1. Implement greedy generation without a cache as the reference, and a KV-cached decoder that
   produces identical tokens and logits within a stated tolerance.
2. Derive and implement exact integer formulas for multiply-adds per generated token and for
   KV-cache bytes, and verify both against counts and array sizes the running code measures.
3. Implement left-padded static batching with a padding mask and per-row position ids, and
   measure how many requests change their output when either one is wrong.
4. Compute static batching's padding and idle waste, and its time to first token, inter-token
   latency and throughput in logical steps, from an arrival schedule.
5. Measure the wall-clock speed-up of the cache at four sequence lengths, printed with units and
   labelled machine-dependent, next to the exactly counted operation ratio it approximates.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `T03-L03`

**This lesson assumes you can already:**

- write Python functions with loops and index and slice numpy arrays, including with boolean masks
- predict the shape that a numpy matmul or broadcast of batched arrays will return, such as (batch, heads, seq, dim) against (batch, heads, dim, seq)
- time a function with time.perf_counter and say why two runs of the same code give different times

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Mastering LLM Techniques: Inference Optimization](https://developer.nvidia.com/blog/mastering-llm-techniques-inference-optimization/) · Shashank Verma and Neal Vaidya (NVIDIA Developer Blog) · *free* — the prefill and decode phases, the KV cache and its size formula in bytes, and batching, the three things the lesson counts and measures
- [Text generation (Transformers documentation)](https://huggingface.co/docs/transformers/llm_tutorial) · Hugging Face · *free* — the 'Padding side' pitfall: why a batched prompt must be left-padded, which is the mistake the batching exercise makes you measure
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) · Vaswani et al. (arXiv) · *free* — section 3.2.1 defines scaled dot-product attention and 3.2.3 the masking (to minus infinity) that stops a token seeing later positions, which attend and the cached decoder both compute
- [Broadcasting (NumPy user guide)](https://numpy.org/doc/stable/user/basics.broadcasting.html) · NumPy developers · *free* — the broadcasting rules behind the padding mask and per-row position ids, so a shape error in a batched array is quick to read

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import os
import signal
import sys
import threading
import time
import traceback
from typing import Any, Callable, NamedTuple, Sequence

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)

SEED = 20260929          # this lesson's one fixed seed: the weights, every prompt and the
                          # request workload derive from it, so re-running prints the same numbers
LOGIT_TOLERANCE = 1e-9    # "identical logits" means: equal to within this, in float64

_FAILED_CHECKS: list[str] = []



# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("attend",),
    "exercise 2": ("generate_uncached",),
    "exercise 3": ("cache_append", "cache_view"),
    "exercise 4": ("decode_step",),
    "exercise 5": ("forward_macs", "generation_macs"),
    "exercise 6": ("kv_cache_bytes", "max_sequences"),
    "exercise 7": ("make_batch_inputs",),
    "exercise 8": ("static_batch_waste",),
    "exercise 9": ("static_schedule",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (cache_append, cache_view)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 30     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 2048     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The model you are serving

It is a decoder-only transformer in the family of the nanoLM you trained in T03-L03 (token plus
position embeddings, causal self-attention, residual connections, a linear head), extended by
exactly the parts that serving costs depend on: several layers, several attention heads, and an
MLP in every layer. Each layer is

```
x  = x + Wo · attention( norm(x) )        # pre-norm, causal, n_heads heads of head_dim each
x  = x + W2 · gelu( W1 · norm(x) )        # the MLP, d_ff wide
```

and the last position's hidden state goes through one more norm and the output head to give a
score (a logit) for every token in the vocabulary. Generation is greedy: at each step take the
highest-scoring token, append it, and go again.

Two conventions the whole lesson leans on. **Multiply-adds:** every matrix product goes through
the given `mm(a, b)`, which counts its multiply-adds exactly (`a.size × b.shape[-1]`), so that
section 7 can compare a formula with what the code really did. **Shapes:** activations are
`(batch, positions, d_model)`; per-head tensors are `(batch, heads, positions, head_dim)`.

Everything below is given to you. Read it, then run it.

In [ ]:
class ModelConfig(NamedTuple):
    vocab: int      # number of distinct token ids
    d_model: int    # width of the residual stream
    n_layers: int
    n_heads: int
    d_ff: int       # hidden width of each layer's MLP
    max_pos: int    # number of learned position embeddings

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads


CFG = ModelConfig(vocab=64, d_model=64, n_layers=2, n_heads=4, d_ff=256, max_pos=512)

_macs = 0


def mm(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """`a @ b`, counting its multiply-adds. For a (..., m, k) @ (..., k, n) product the count is
    batch × m × k × n, which is exactly `a.size × n`. Every matrix product in this lesson goes
    through here, so what it counts is what the code actually multiplied."""
    global _macs
    _macs += int(a.size) * int(b.shape[-1])
    return a @ b


def measure_macs(fn: Callable[..., Any], *args: Any, **kwargs: Any) -> tuple[Any, int]:
    """Run `fn` and return (its result, the exact number of multiply-adds `mm` counted)."""
    before = _macs
    result = fn(*args, **kwargs)
    return result, _macs - before


def softmax(x: np.ndarray) -> np.ndarray:
    """Softmax over the last axis, shifted by the row maximum so large scores cannot overflow."""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def rmsnorm(x: np.ndarray) -> np.ndarray:
    """Scale each vector to unit root-mean-square (no learned gain, to keep the model small)."""
    return x / np.sqrt(np.mean(x * x, axis=-1, keepdims=True) + 1e-6)


def gelu(x: np.ndarray) -> np.ndarray:
    return 0.5 * x * (1.0 + np.tanh(0.7978845608028654 * (x + 0.044715 * x ** 3)))


def split_heads(x: np.ndarray, n_heads: int) -> np.ndarray:
    """(B, T, D) -> (B, H, T, D/H)."""
    b, t, d = x.shape
    return x.reshape(b, t, n_heads, d // n_heads).transpose(0, 2, 1, 3)


def merge_heads(x: np.ndarray) -> np.ndarray:
    """(B, H, T, dh) -> (B, T, H*dh)."""
    b, h, t, dh = x.shape
    return x.transpose(0, 2, 1, 3).reshape(b, t, h * dh)


def init_params(cfg: ModelConfig, seed: int) -> dict:
    """Random float64 weights from a seeded generator. The scales are fixed choices of this
    lesson (unit-scale embeddings, 1/sqrt(fan-in) matrices, a sharpened head) that make the
    greedy path varied instead of degenerate; nothing was tuned or searched."""
    rng = np.random.default_rng(seed)
    d = cfg.d_model
    layers = []
    for _ in range(cfg.n_layers):
        layers.append({
            "wq": rng.normal(0, 1 / np.sqrt(d), (d, d)), "wk": rng.normal(0, 1 / np.sqrt(d), (d, d)),
            "wv": rng.normal(0, 1 / np.sqrt(d), (d, d)), "wo": rng.normal(0, 1 / np.sqrt(d), (d, d)),
            "w1": rng.normal(0, 1 / np.sqrt(d), (d, cfg.d_ff)),
            "w2": rng.normal(0, 1 / np.sqrt(cfg.d_ff), (cfg.d_ff, d)),
        })
    return {"tok": rng.normal(0, 1, (cfg.vocab, d)), "pos": rng.normal(0, 1, (cfg.max_pos, d)),
            "head": rng.normal(0, 3 / np.sqrt(d), (d, cfg.vocab)), "layers": layers}


PARAMS = init_params(CFG, SEED)


def project_qkv(x: np.ndarray, w: dict, cfg: ModelConfig):
    """Norm, then the three projections, split into heads. Returns (q, k, v), each (B, H, T, dh)."""
    a = rmsnorm(x)
    return tuple(split_heads(mm(a, w[name]), cfg.n_heads) for name in ("wq", "wk", "wv"))


def finish_block(x: np.ndarray, context: np.ndarray, w: dict) -> np.ndarray:
    """Everything in a layer AFTER attention: output projection and residual, then the MLP and
    its residual. `context` is attention's output, (B, H, T, dh)."""
    x = x + mm(merge_heads(context), w["wo"])
    return x + mm(gelu(mm(rmsnorm(x), w["w1"])), w["w2"])


def lm_head(params: dict, x_last: np.ndarray) -> np.ndarray:
    """Final norm and the output head on ONE position per row: (B, D) -> (B, vocab) logits."""
    return mm(rmsnorm(x_last), params["head"])


_n_params = sum(int(a.size) for a in (PARAMS["tok"], PARAMS["pos"], PARAMS["head"])) + sum(
    int(a.size) for w in PARAMS["layers"] for a in w.values())
print(f"config: {CFG.n_layers} layers, {CFG.n_heads} heads of {CFG.head_dim}, d_model {CFG.d_model}, "
      f"d_ff {CFG.d_ff}, vocab {CFG.vocab}")
print(f"the model has {_n_params:,} parameters ({_n_params * 8 / 2**20:.2f} MiB in float64)")

## 2. Exercise 1 — `attend(q, k, v, mask)`

Attention is the only place where positions talk to each other, so it is the only place a cache
and a padding mask can go wrong. For each query, score every key, turn the scores into weights
that sum to one, and return the weighted sum of the values (Vaswani et al., see `claims.yaml`,
for the scaling this lesson uses).

`mask` says which keys each query may use: `True` means "may attend", and it broadcasts against
the `(B, H, Tq, Tk)` scores, so a `(B, 1, 1, Tk)` padding mask and a `(1, 1, T, T)` causal mask
both work. `None` means every key is allowed. A masked key must get weight **exactly** zero
whenever its query has at least one legal key. A query with NO legal key (a padding position
has nothing to look at) must still return finite numbers: nothing real ever reads them, but a
`nan` there would spread into everything that follows.

<details><summary>💡 Hint 1 — what to think about</summary>

Shapes first: the query block `(Tq, dh)` against the key block `(Tk, dh)` gives a `(Tq, Tk)`
table of scores per head. Which dimension is the scale, `head_dim` or `d_model`? And what
should a masked score become BEFORE the softmax, so that its weight after it is zero?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Multiply queries by the transposed keys, divide by the square root of the head width, replace
the scores of disallowed keys by a number so negative that its exponential is zero, take the
softmax over the key axis, and multiply by the values. Use `mm` for both products. Ask yourself
what softmax does with a row where every entry is the same "very negative" number, and what it
does with a row of `-inf`.
</details>

In [ ]:
def attend(q: np.ndarray, k: np.ndarray, v: np.ndarray, mask) -> np.ndarray:
    """Masked scaled dot-product attention.

    q: (B, H, Tq, dh).  k, v: (B, H, Tk, dh).  mask: None, or a bool array broadcastable to
    (B, H, Tq, Tk), True where the query may use that key. Returns (B, H, Tq, dh).

    Worked example (one head, one query, two keys, head width 2, no mask):
        q = [[1, 0]], k = [[1, 0], [0, 1]], v = [[10, 0], [0, 10]]
        scores = q·kᵀ / sqrt(2) = [0.707, 0.0]  ->  weights ≈ [0.670, 0.330]
        output = weights @ v ≈ [[6.70, 3.30]]
    With mask [[True, False]] the second key is blocked and the output is exactly [[10, 0]].
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_attend() -> None:
    q = np.array([[[[1.0, 0.0]]]])
    k = np.array([[[[1.0, 0.0], [0.0, 1.0]]]])
    v = np.array([[[[10.0, 0.0], [0.0, 10.0]]]])
    out = attend(q, k, v, None)
    assert out.shape == q.shape, f"returned shape {out.shape}, expected {q.shape}"
    weights = np.exp(np.array([1.0, 0.0]) / np.sqrt(2.0))
    weights /= weights.sum()
    assert np.allclose(out[0, 0, 0], weights @ v[0, 0], rtol=0, atol=1e-9), (
        "output does not match softmax(q·kᵀ / sqrt(head_dim)) @ v: check the scale is "
        "sqrt(head_dim) (here 2), not sqrt(d_model), and that softmax runs over the KEY axis")
    # Two heads of width 3 against 5 keys, so the head width, the number of keys and the model
    # width (2 × 3) all differ and a wrong scale cannot hide behind a coincidence.
    rng = np.random.default_rng(1)
    q2, k2, v2 = rng.normal(size=(1, 2, 4, 3)), rng.normal(size=(1, 2, 5, 3)), rng.normal(size=(1, 2, 5, 3))
    s = np.einsum("bhqd,bhkd->bhqk", q2, k2) / np.sqrt(3)
    w2 = np.exp(s - s.max(axis=-1, keepdims=True))
    want2 = np.einsum("bhqk,bhkd->bhqd", w2 / w2.sum(axis=-1, keepdims=True), v2)
    gap2 = float(np.max(np.abs(attend(q2, k2, v2, None) - want2)))
    assert gap2 < 1e-9, (
        f"with 2 heads of width 3 and 5 keys the output is off by {gap2:.1e}: the scale is the square "
        "root of ONE head's width, q.shape[-1], not of the number of keys or of heads × width")
    blocked = attend(q, k, v, np.array([[[[True, False]]]]))
    assert np.allclose(blocked[0, 0, 0], v[0, 0, 0], rtol=0, atol=1e-9), (
        "a masked key still received weight: set masked scores to a large negative BEFORE "
        "the softmax (multiplying the weights by the mask afterwards does not renormalise)")
    empty = attend(q, k, v, np.zeros((1, 1, 1, 2), dtype=bool))
    assert np.isfinite(empty).all(), (
        "a query with EVERY key masked produced nan/inf. Masking with -inf gives (-inf) - (-inf) "
        "inside the softmax; use a large finite negative instead")
    big = attend(q * 1e3, k * 1e3, v, None)
    assert np.isfinite(big).all(), "huge scores overflowed: subtract the row maximum in the softmax"
    _, macs = measure_macs(attend, q, k, v, None)
    assert macs == 2 * 1 * 1 * 1 * 2 * 2, (
        f"attend counted {macs} multiply-adds for a 1-query, 2-key, width-2 head; expected 8 "
        "(scores and weighted sum, 4 each). Did a product bypass mm()?")
    print("attend: weights, masking, fully masked rows and multiply-add count all check out")


if __name__ == "__main__":
    _try("exercise 1", _check_attend)

## 3. The reference: generate without a cache

`forward_full` runs the whole model on a block of token ids. It applies a causal mask (position
`i` may look at positions `<= i`, and at nothing else), optionally a padding mask and explicit
position ids (section 10 needs both), and returns the logits of the LAST position of each row.
It is built from your `attend`, so it needs exercise 1.

Serving without a cache is then almost embarrassingly simple, and that is the point: to get the
next token, run `forward_full` on the entire sequence so far, read the last logits, append the
arg-max, repeat. Every step recomputes every earlier position's keys and values from scratch.

In [ ]:
def forward_full(params: dict, cfg: ModelConfig, tokens, positions=None, key_mask=None, cache=None):
    """The whole model on `tokens` (B, T). Returns the logits of the last position, (B, vocab).

    positions: (B, T) position ids; default 0..T-1 in every row.
    key_mask:  (B, T) bool, False marks a padding slot nobody may attend to; default all True.
    cache:     if given, every layer's keys and values for all T positions are appended to it
               (this is the "prefill" of section 5).
    """
    tokens = np.asarray(tokens)
    b, t = tokens.shape
    if positions is None:
        positions = np.broadcast_to(np.arange(t), (b, t))
    if int(np.max(positions)) >= cfg.max_pos:
        raise ValueError(f"position {int(np.max(positions))} is past the {cfg.max_pos} this model has")
    x = params["tok"][tokens] + params["pos"][positions]
    allowed = np.tril(np.ones((t, t), dtype=bool))[None, None]          # causal: (1, 1, T, T)
    if key_mask is not None:
        allowed = allowed & np.asarray(key_mask)[:, None, None, :]      # and no padding keys
    for layer_index, w in enumerate(params["layers"]):
        q, k, v = project_qkv(x, w, cfg)
        if cache is not None:
            cache_append(cache, layer_index, k, v)
        x = finish_block(x, attend(q, k, v, allowed), w)
    return lm_head(params, x[:, -1, :])

## 4. Exercise 2 — `generate_uncached(params, cfg, prompt, n_new)`

Write the loop above. Return BOTH the new tokens (not the prompt) and the logits that chose
them, one row per generated token: section 6 compares those logits, not just the tokens,
because two decoders can agree on every arg-max and still disagree about the numbers.

<details><summary>💡 Hint 1 — what to think about</summary>

What is the input of each step, and how long is it? Which row of the result do you read the
next token from, and what should happen to the caller's `prompt` list?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Keep a growing list of ids that starts as a copy of the prompt. Each step: wrap it as a
one-row array, call `forward_full`, remember the logits row, append the index of its largest
entry as a plain `int`. When done, return the appended part and the stacked logits, shaped
`(n_new, vocab)` even when `n_new` is 0.
</details>

In [ ]:
def generate_uncached(params: dict, cfg: ModelConfig, prompt: Sequence[int], n_new: int):
    """Greedy decoding that recomputes the whole sequence at every step.

    Returns (new_tokens, logits): a list of `n_new` python ints, and a float array of shape
    (n_new, vocab) whose row j holds the scores that chose token j. `prompt` is not modified.

    Worked example: with a 3-token prompt and n_new = 2 the first step runs the model on 3
    positions, the second on 4 (the prompt plus the token just chosen); the result is a list of
    2 ints and an array of shape (2, vocab).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _make_prompt(length: int, seed: int) -> list[int]:
    """A seeded prompt of random token ids. Given to you."""
    return np.random.default_rng(seed).integers(0, CFG.vocab, length).tolist()


def _check_generate_uncached() -> None:
    prompt = _make_prompt(6, SEED + 1)
    before = list(prompt)
    with _time_limit(10.0, "this call", _MEMORY_MIB):
        tokens, logits = generate_uncached(PARAMS, CFG, prompt, 5)
    assert prompt == before, "the caller's prompt list was modified: work on a copy"
    assert len(tokens) == 5 and all(type(t) is int for t in tokens), (
        f"expected a list of 5 plain ints (the NEW tokens only), got {tokens!r}")
    assert logits.shape == (5, CFG.vocab), f"logits shaped {logits.shape}, expected (5, {CFG.vocab})"
    for j, t in enumerate(tokens):
        assert t == int(np.argmax(logits[j])), (
            f"token {j} is not the arg-max of its own logits row: append the arg-max, "
            "and remember the row that chose it")
    for j in range(len(tokens)):
        whole = forward_full(PARAMS, CFG, np.array([prompt + tokens[:j]]))[0]
        assert np.allclose(logits[j], whole, rtol=0, atol=LOGIT_TOLERANCE), (
            f"logits row {j} is not what forward_full gives over the prompt plus the {j} tokens "
            "chosen so far: feed the WHOLE sequence to forward_full at every step, and keep the "
            "row that chose each token")
    with _time_limit(10.0, "this call", _MEMORY_MIB):
        none, none_logits = generate_uncached(PARAMS, CFG, prompt, 0)
    assert none == [] and none_logits.shape == (0, CFG.vocab), "n_new = 0 must return ([], (0, vocab))"
    print(f"generate_uncached: {len(tokens)} tokens, each the arg-max of the logits that chose it")


if __name__ == "__main__":
    _try("exercise 2", _check_generate_uncached, needs=("exercise 1",))

In [ ]:
def _show_uncached_cost() -> None:
    """How much arithmetic does each extra token cost when nothing is remembered? Measured."""
    prompt = _make_prompt(8, SEED + 2)
    totals = {0: 0}
    for n in (1, 2, 15, 16, 31, 32):
        (_, _), totals[n] = measure_macs(generate_uncached, PARAMS, CFG, prompt, n)
    print(f"prompt of {len(prompt)} tokens; multiply-adds counted by mm() while generating:")
    for step in (1, 2, 16, 32):
        print(f"  step {step:2d} alone (a forward pass over {len(prompt) + step - 1:2d} positions): "
              f"{totals[step] - totals[step - 1]:>9,} multiply-adds")
    print("each step reruns a longer sequence, so the cost of the next token keeps growing")


if __name__ == "__main__":
    _try("uncached cost", _show_uncached_cost, needs=("exercise 2",))

## 5. The KV cache, and Exercise 3 — `cache_append` and `cache_view`

Look at what the uncached loop throws away. Position `i`'s keys and values at layer `l` depend
only on tokens `0..i` (that is what the causal mask guarantees), so when a new token arrives
they have not changed, and recomputing them is pure waste. A **KV cache** keeps them. Each new
token then needs its own query, key and value only: append the new key and value, and let the
one new query attend over everything stored.

The cache is one preallocated array pair per model, shaped
`(layers, batch, heads, max_len, head_dim)`, plus a fill counter per layer. You write the two
operations on it; the class around them is given.

- `cache_append(cache, layer, k_new, v_new)` writes `k_new`, `v_new` (each `(B, H, m, dh)`) into
  that layer's slots after the ones already filled, and advances THAT layer's counter by `m`.
  It raises `ValueError` if they would not fit in `max_len`.
- `cache_view(cache, layer)` returns that layer's filled keys and values, `(B, H, filled, dh)`
  each, and nothing beyond `filled`.

<details><summary>💡 Hint 1 — what to think about</summary>

Each layer has its own fill counter because layers are filled one after another inside a single
step. What would go wrong if one shared counter were advanced after layer 0's append? And what
should a caller see in the slots that have not been filled yet?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Append: read the layer's counter, work out where the new block ends, refuse if that is past
`max_len`, assign into a slice of `cache.k` and of `cache.v`, then advance the counter. View:
slice up to the counter. Return the slice itself rather than a copy: a copy of the whole cache
on every step would give back part of what the cache saves.
</details>

In [ ]:
class KVCache:
    """Preallocated key/value storage for `batch` sequences of up to `max_len` positions."""

    def __init__(self, cfg: ModelConfig, batch: int, max_len: int, dtype=np.float64):
        shape = (cfg.n_layers, batch, cfg.n_heads, max_len, cfg.head_dim)
        self.k = np.zeros(shape, dtype=dtype)
        self.v = np.zeros(shape, dtype=dtype)
        self.lengths = [0] * cfg.n_layers      # filled positions, per layer
        self.max_len = max_len

    @property
    def length(self) -> int:
        """Positions stored, once every layer has been filled for them."""
        return self.lengths[0]

    def nbytes_allocated(self) -> int:
        return int(self.k.nbytes + self.v.nbytes)

    def nbytes_used(self) -> int:
        """Bytes of the filled part only, measured from the arrays `cache_view` hands back."""
        total = 0
        for layer in range(len(self.lengths)):
            keys, values = cache_view(self, layer)
            total += int(keys.nbytes + values.nbytes)
        return total


def cache_append(cache: KVCache, layer: int, k_new: np.ndarray, v_new: np.ndarray) -> None:
    """Store `k_new`, `v_new` (B, H, m, dh) after the positions `layer` already holds.

    Worked example: a fresh cache with max_len 4, then append m = 3 -> layer's counter is 3;
    append m = 1 -> 4; append m = 1 again raises ValueError (it would be position 5 of 4).
    Other layers' counters are not touched.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def cache_view(cache: KVCache, layer: int):
    """The filled keys and values of `layer`: two arrays of shape (B, H, filled, dh).

    Worked example: after appending 3 positions to layer 0, cache_view(cache, 0) returns keys
    and values whose third axis has length 3, even though max_len is larger.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_cache() -> None:
    small = ModelConfig(vocab=8, d_model=8, n_layers=2, n_heads=2, d_ff=8, max_pos=16)
    cache = KVCache(small, batch=2, max_len=4)
    rng = np.random.default_rng(0)
    first = rng.normal(size=(2, 2, 3, 4)), rng.normal(size=(2, 2, 3, 4))
    cache_append(cache, 0, *first)
    assert cache.lengths == [3, 0], (
        f"after one append to layer 0 the counters are {cache.lengths}, expected [3, 0]: "
        "each layer keeps its own counter, and only the layer appended to advances")
    keys, values = cache_view(cache, 0)
    assert keys.shape == (2, 2, 3, 4), f"view has shape {keys.shape}: it must stop at the filled length"
    assert np.array_equal(keys, first[0]) and np.array_equal(values, first[1]), (
        "the view does not hold what was appended (keys and values swapped, or the wrong slice?)")
    second = rng.normal(size=(2, 2, 1, 4)), rng.normal(size=(2, 2, 1, 4))
    cache_append(cache, 0, *second)
    keys, _ = cache_view(cache, 0)
    assert np.array_equal(keys[:, :, 3:], second[0]) and np.array_equal(keys[:, :, :3], first[0]), (
        "a second append must land AFTER the first, leaving the first block untouched")
    try:
        cache_append(cache, 0, *second)
    except ValueError:
        pass
    else:
        raise AssertionError("appending a 5th position to a cache of max_len 4 must raise ValueError")
    assert cache.lengths[0] == 4 and np.array_equal(cache_view(cache, 0)[0][:, :, 3:], second[0]), (
        "a refused append must leave the cache exactly as it was: check for room BEFORE writing "
        "or advancing the counter")
    assert cache_view(cache, 1)[0].shape[2] == 0, "layer 1 was never appended to and must be empty"
    assert np.shares_memory(cache_view(cache, 0)[0], cache.k), (
        "cache_view returned a copy; return a slice of cache.k so nothing is copied per step")
    print("cache_append / cache_view: order, per-layer counters, the full-cache error and no-copy views")


if __name__ == "__main__":
    _try("exercise 3", _check_cache)

## 6. Exercise 4 — `decode_step(params, cfg, cache, tokens, positions, key_mask)`

This is the cached decoder's inner loop, and the heart of the lesson. It processes ONE new token
per row, given the cache that holds everything before it:

1. embed the token and its position (`tokens` and `positions` are `(B,)` arrays);
2. for every layer: `project_qkv` on that one position, `cache_append` its key and value, then
   `cache_view` and let the single query `attend` over everything stored, itself included;
   `finish_block` closes the layer;
3. `lm_head` on the last (only) position gives the `(B, vocab)` logits.

`key_mask`, when given, is `(B, T)` with `T` the cache length AFTER this step's append; `False`
marks a slot no query may use (padding, section 10). One query needs no causal mask: everything
in the cache is in its past.

<details><summary>💡 Hint 1 — what to think about</summary>

At which point in the layer does the new key and value have to enter the cache, so that the new
query can see its own position? What is the shape of the query, and therefore of the score
table? How does a `(B, T)` padding mask need reshaping to broadcast against it?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Add a length-1 position axis to the embeddings so the layer helpers see `(B, 1, D)`. In each
layer project, append to the cache, read the cache back, attend with the padding mask (or none)
reshaped to `(B, 1, 1, T)`, and finish the block. At the end read logits from position 0 of the
single-position result. Use only the helpers named above so every multiply-add is counted.
</details>

In [ ]:
def decode_step(params: dict, cfg: ModelConfig, cache: KVCache, tokens, positions, key_mask=None):
    """Advance every row by ONE token using (and extending) the cache. Returns (B, vocab) logits.

    tokens: (B,) ids of the newest token of each row.  positions: (B,) its position id.
    key_mask: None, or (B, T) bool over the cache slots AFTER appending (False = padding).

    Worked example: after a prefill of 5 positions, decode_step with one token leaves every
    layer's counter at 6, and its multiply-adds are those of ONE position attending over 6.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def generate_cached(params: dict, cfg: ModelConfig, prompt: Sequence[int], n_new: int,
                    cache_dtype=np.float64):
    """Greedy decoding with a KV cache. Same return as `generate_uncached`. Given to you.

    One prefill: `forward_full` over the whole prompt fills the cache. Then each new token costs
    one `decode_step`. The very last token chosen is never fed back, so it is never cached."""
    prompt = list(prompt)
    if n_new == 0:
        return [], np.zeros((0, cfg.vocab))
    cache = KVCache(cfg, batch=1, max_len=len(prompt) + n_new, dtype=cache_dtype)
    logits = forward_full(params, cfg, np.array([prompt]), cache=cache)[0]
    tokens, rows = [], []
    for step in range(n_new):
        rows.append(logits)
        tokens.append(int(np.argmax(logits)))
        if step < n_new - 1:
            position = len(prompt) + step
            logits = decode_step(params, cfg, cache, np.array([tokens[-1]]), np.array([position]))[0]
    return tokens, np.array(rows)


def _check_decode_step() -> None:
    prompt = _make_prompt(7, SEED + 3)
    n_new = 6
    want, want_logits = generate_uncached(PARAMS, CFG, prompt, n_new)
    got, got_logits = generate_cached(PARAMS, CFG, prompt, n_new)
    assert got == want, (
        f"cached tokens {got} differ from uncached {want}. A cache changes speed, never results: "
        "check the new key/value is appended BEFORE the query attends, that the position id is "
        "the token's real position, and that attention reads the cache, not just the new token")
    gap = float(np.max(np.abs(got_logits - want_logits)))
    assert gap < LOGIT_TOLERANCE, (
        f"tokens agree but the logits differ by {gap:.1e}, over the {LOGIT_TOLERANCE:.0e} "
        "tolerance: something differs between the two decoders (attending before the append? "
        "a float32 cache? a wrong scale in attend, which both decoders use?)")
    cache = KVCache(CFG, batch=1, max_len=len(prompt) + 2)
    forward_full(PARAMS, CFG, np.array([prompt]), cache=cache)
    decode_step(PARAMS, CFG, cache, np.array([want[0]]), np.array([len(prompt)]))
    assert cache.lengths == [len(prompt) + 1] * CFG.n_layers, (
        f"after one step every layer should hold {len(prompt) + 1} positions, "
        f"got {cache.lengths}: append exactly once per layer per step")
    # Two rows of different lengths, left-padded by hand: a 3-token prompt behind 4 pad slots
    # and a 7-token one. Each row's next token sits at its OWN position and the pads are masked.
    rows = [_make_prompt(3, SEED + 60), _make_prompt(7, SEED + 61)]
    tokens = np.zeros((2, 7), dtype=np.int64)
    positions = np.zeros((2, 7), dtype=np.int64)
    keep = np.zeros((2, 7), dtype=bool)
    for r, ids in enumerate(rows):
        tokens[r, 7 - len(ids):], positions[r, 7 - len(ids):], keep[r, 7 - len(ids):] = ids, np.arange(len(ids)), True
    cache = KVCache(CFG, batch=2, max_len=8)
    forward_full(PARAMS, CFG, tokens, positions, keep, cache=cache)
    new = np.array([11, 22])
    mask_after = np.concatenate([keep, np.ones((2, 1), dtype=bool)], axis=1)
    got = decode_step(PARAMS, CFG, cache, new, np.array([3, 7]), mask_after)
    want_rows = forward_full(PARAMS, CFG, np.concatenate([tokens, new[:, None]], axis=1),
                             np.concatenate([positions, np.array([[3], [7]])], axis=1), mask_after)
    assert np.allclose(got, want_rows, rtol=0, atol=LOGIT_TOLERANCE), (
        "with two left-padded rows the cached step disagrees with recomputing them: each row's new "
        "token must use ITS OWN position id (the `positions` argument, not the cache length), and "
        "`key_mask` must reach attend as key_mask[:, None, None, :] so pad slots get no weight")
    print(f"decode_step: {n_new} cached tokens equal the uncached ones and the logits agree "
          f"to within {LOGIT_TOLERANCE:.0e}")


if __name__ == "__main__":
    _try("exercise 4", _check_decode_step, needs=("exercise 1", "exercise 2", "exercise 3"))

In [ ]:
def _compare_cached_and_uncached() -> None:
    """Eight different prompts, greedy, cached against uncached: tokens, logits and how close a
    call any arg-max was, so that "identical" cannot be luck."""
    same_tokens, worst_gap, close_calls = 0, 0.0, 0
    n_new = 48
    for i in range(8):
        prompt = _make_prompt(4 + 3 * i, SEED + 10 + i)
        a, la = generate_uncached(PARAMS, CFG, prompt, n_new)
        b, lb = generate_cached(PARAMS, CFG, prompt, n_new)
        same_tokens += int(a == b)
        worst_gap = max(worst_gap, float(np.max(np.abs(la - lb))))
        top_two = np.sort(la, axis=1)[:, -2:]
        close_calls += int(np.sum(top_two[:, 1] - top_two[:, 0] < 1e-6))
    print(f"{same_tokens} of 8 prompts: cached and uncached greedy tokens identical over {n_new} steps each")
    print(f"logits within the {LOGIT_TOLERANCE:.0e} tolerance on every step: {worst_gap < LOGIT_TOLERANCE}")
    print(f"steps where the best and second-best token were within 1e-6 (where rounding could flip "
          f"a choice): {close_calls} of {8 * n_new}")


if __name__ == "__main__":
    _try("cached vs uncached", _compare_cached_and_uncached, needs=("exercise 4",))

## 7. Exercise 5 — counting the arithmetic exactly

"The cache is faster" is a claim about arithmetic, so count it. One forward pass over `new`
positions, with `cached` positions already stored, costs (in multiply-adds, counting matrix
products only, and counting what this dense numpy code really multiplies):

- per layer: the four `d_model × d_model` projections (`q`, `k`, `v` and the output `wo`) for
  each of the `new` positions; the score products, `new` queries against all `cached + new`
  keys, `d_model` multiply-adds each; the same again for the weighted sum of values; and the two
  MLP products for each new position;
- once: the output head on the LAST position only, `d_model × vocab`.

Softmax, norms and the GELU are elementwise and not counted. The score block is dense: the
masked half of a causal `new × new` block is multiplied all the same, and a kernel that skips
it would do less. You write `forward_macs`, and `generation_macs` that adds it up over a whole
generation, with and without the cache. The formulas are checked against `mm`'s own count.

<details><summary>💡 Hint 1 — what to think about</summary>

Per layer, how many `d × d` products are there for each new position, and how many `d × d_ff`
ones? How many (query, key) pairs does the score table hold when `new` queries meet
`cached + new` keys, and how many multiply-adds does each pair cost across all heads?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Add the four pieces per layer, multiply by the number of layers, add the head once. For a whole
generation the uncached step that produces token `j` runs `forward_macs` over the prompt plus
the `j` tokens chosen before it, with nothing cached. The cached run does one prefill over the
prompt and then one-position steps, each with everything before it cached.
</details>

In [ ]:
def forward_macs(cfg: ModelConfig, cached: int, new: int) -> int:
    """Multiply-adds of one forward pass over `new` positions with `cached` already stored
    (new >= 1). Exact, an int.

    Worked example: a model with 1 layer, d_model 2, d_ff 3, vocab 5, run on new = 1 position
    with cached = 0: projections 4·1·2·2 = 16, scores and values 2·2·1·(0+1) = 4, MLP
    2·1·2·3 = 12, so 32 in the layer, plus the head 2·5 = 10: 42 in total.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def generation_macs(cfg: ModelConfig, prompt_len: int, n_new: int, use_cache: bool) -> int:
    """Total multiply-adds to generate `n_new` tokens after a `prompt_len`-token prompt.

    Uncached: the step that makes token j (j = 0 ... n_new-1) runs over prompt_len + j positions.
    Cached: one prefill over the prompt, then n_new - 1 one-position steps (the last token chosen
    is never fed back). n_new = 0 costs 0.

    Worked example: n_new = 1 costs the same with or without the cache, one pass over the prompt;
    from n_new = 2 on the cached run is cheaper.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_macs() -> None:
    with _time_limit(10.0, "this call", _MEMORY_MIB):
        _check_macs_inner()


def _check_macs_inner() -> None:
    tiny = ModelConfig(vocab=5, d_model=2, n_layers=1, n_heads=1, d_ff=3, max_pos=8)
    assert forward_macs(tiny, 0, 1) == 42, (
        f"forward_macs gave {forward_macs(tiny, 0, 1)} for the worked example in its docstring, "
        "which adds up to 42: recount projections, scores + values, MLP, then the head once")
    prompt = _make_prompt(8, SEED + 4)
    for use_cache, generate in ((False, generate_uncached), (True, generate_cached)):
        for n_new in (1, 5):
            (_, _), measured = measure_macs(generate, PARAMS, CFG, prompt, n_new)
            formula = generation_macs(CFG, len(prompt), n_new, use_cache)
            kind = "cached" if use_cache else "uncached"
            assert measured == formula, (
                f"{kind}, {n_new} new tokens: mm() counted {measured:,} multiply-adds but your "
                f"formula says {formula:,}. Recheck the formula term by term (one forward pass "
                "at a time); if the formula is right for one pass, the cached decoder may not be "
                "using project_qkv, cache_view, attend and finish_block from this lesson")
    assert generation_macs(CFG, 8, 0, True) == generation_macs(CFG, 8, 0, False) == 0
    assert generation_macs(CFG, 8, 1, True) == generation_macs(CFG, 8, 1, False), (
        "one new token costs the same with or without the cache: the cached run is one prefill")
    print("forward_macs and generation_macs equal what mm() counted, for both decoders")


if __name__ == "__main__":
    _try("exercise 5", _check_macs, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

In [ ]:
def _show_op_ratio() -> None:
    """The exact arithmetic ratio between the two decoders, for growing generations."""
    prompt_len = 8
    print(f"prompt of {prompt_len} tokens; multiply-adds to generate n new tokens")
    print(f"{'n':>5}  {'uncached':>16}  {'cached':>14}  {'uncached / cached':>18}")
    for n in (1, 8, 32, 128, 256, 400):
        u = generation_macs(CFG, prompt_len, n, False)
        c = generation_macs(CFG, prompt_len, n, True)
        print(f"{n:>5}  {u:>16,}  {c:>14,}  {u / c:>12.1f} times")
    per_token = forward_macs(CFG, 300, 1)
    dense = forward_macs(CFG, 0, 301)
    print(f"the 301st position: {per_token:,} multiply-adds with a cache, {dense:,} without "
          f"({dense / per_token:.1f} times)")
    print("the ratio keeps growing with n: cached cost per token grows only through attention over "
          "the cache, uncached cost grows through everything")


if __name__ == "__main__":
    _try("op ratio", _show_op_ratio, needs=("exercise 5",))

## 8. Exercise 6 — what the cache costs in memory

The cache trades arithmetic for memory, and the memory is the part that limits how many
requests a server can hold at once (Kwon et al., see `claims.yaml`, build a whole serving system
around managing exactly this memory). For one sequence it stores a key vector and a value vector
of `head_dim` numbers, for every head, at every layer, at every position:

`bytes = 2 × layers × heads × head_dim × positions × batch × bytes_per_number`

The 2 is keys and values. Note what is NOT in it: the vocabulary size, `d_ff`, and the number
of parameters. Write the formula as `kv_cache_bytes`, then use it in `max_sequences` to answer
the question a server actually asks: how many full-length sequences fit in a memory budget?

<details><summary>💡 Hint 1 — what to think about</summary>

Which of the model's numbers appear in the cache and which do not? Is `heads × head_dim` a
number you already have under another name? For the second function: is the answer a whole
number, and what happens when the budget is one byte short of the next sequence?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Multiply the factors of the formula above. For `max_sequences`, the bytes one sequence needs
come from the first function, and a budget holds a whole number of them, no more.
</details>

In [ ]:
def kv_cache_bytes(cfg: ModelConfig, positions: int, batch: int = 1, itemsize: int = 8) -> int:
    """Bytes of a KV cache holding `positions` positions for `batch` sequences, each number
    taking `itemsize` bytes (8 for float64, 4 for float32, 2 for float16). An exact int.

    Worked example: 1 layer, 2 heads of width 4, 10 positions, 1 sequence, float32:
    2 · 1 · 2 · 4 · 10 · 1 · 4 = 640 bytes.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def max_sequences(cfg: ModelConfig, budget_bytes: int, seq_len: int, itemsize: int = 8) -> int:
    """How many sequences of `seq_len` positions fit in `budget_bytes` of KV cache (seq_len >= 1).

    Worked example: if one sequence needs 640 bytes, a budget of 1,919 bytes holds 2 and a budget
    of 1,920 holds 3.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_memory() -> None:
    with _time_limit(10.0, "this call", _MEMORY_MIB):
        _check_memory_inner()


def _check_memory_inner() -> None:
    one = ModelConfig(vocab=3, d_model=8, n_layers=1, n_heads=2, d_ff=8, max_pos=16)
    assert kv_cache_bytes(one, 10, 1, 4) == 640, (
        f"got {kv_cache_bytes(one, 10, 1, 4)} for the docstring example, which is 640: "
        "the factors are 2 (keys and values), layers, heads, head_dim, positions, batch, itemsize")
    cache = KVCache(CFG, batch=2, max_len=20)
    prompt = np.array([_make_prompt(12, SEED + 5), _make_prompt(12, SEED + 6)])
    forward_full(PARAMS, CFG, prompt, cache=cache)
    measured = cache.nbytes_used()
    formula = kv_cache_bytes(CFG, positions=12, batch=2, itemsize=cache.k.itemsize)
    assert measured == formula, (
        f"a cache holding 12 positions for 2 sequences measures {measured:,} bytes but your "
        f"formula says {formula:,}: check the factor 2, and that batch and itemsize are used")
    assert kv_cache_bytes(CFG, 0) == 0
    per_sequence = kv_cache_bytes(CFG, 30)
    assert max_sequences(CFG, 3 * per_sequence, 30) == 3 and max_sequences(CFG, 3 * per_sequence - 1, 30) == 2, (
        "a budget of exactly 3 sequences holds 3, and one byte less holds 2: use whole-number "
        "(floor) division, and never round up")
    half = kv_cache_bytes(CFG, 30, 1, 4)
    assert max_sequences(CFG, 3 * half, 30, itemsize=4) == 3, (
        "max_sequences ignored its itemsize argument: pass it on to kv_cache_bytes")
    print("kv_cache_bytes equals the measured size of the arrays; max_sequences counts whole sequences")


if __name__ == "__main__":
    _try("exercise 6", _check_memory, needs=("exercise 3",))

In [ ]:
def _show_memory() -> None:
    """The formula against the arrays, the price of precision, and what it means at scale."""
    length = 200
    print(f"one sequence of {length} positions on this lesson's model:")
    for name, dtype in (("float64", np.float64), ("float32", np.float32)):
        cache = KVCache(CFG, batch=1, max_len=length, dtype=dtype)
        forward_full(PARAMS, CFG, np.array([_make_prompt(length, SEED + 7)]), cache=cache)
        print(f"  {name}: measured {cache.nbytes_used():>8,} bytes; formula "
              f"{kv_cache_bytes(CFG, length, 1, cache.k.itemsize):>8,} bytes")
    a, la = generate_cached(PARAMS, CFG, _make_prompt(8, SEED + 8), 200)
    b, lb = generate_cached(PARAMS, CFG, _make_prompt(8, SEED + 8), 200, cache_dtype=np.float32)
    gap = float(np.max(np.abs(la - lb)))
    rejects = "would reject it" if gap > LOGIT_TOLERANCE else "would accept it"
    tokens = "identical" if a == b else "different"
    print(f"a float32 cache halves the memory. Over 200 greedy steps its logits move by more than "
          f"{LOGIT_TOLERANCE:.0e}: {gap > LOGIT_TOLERANCE}; by more than 1e-3: {gap > 1e-3}; the tokens "
          f"are {tokens}")
    story = ("the tokens and the logits tell different stories" if (a == b) == (gap > LOGIT_TOLERANCE)
             else "here the tokens and the logits agree")
    print(f"so this lesson's 'identical logits' tolerance {rejects}, and {story}: a rounding change "
          "is a decision about quality and deserves its own measurement, not an assumption")
    # A hypothetical shape, chosen only to make the arithmetic vivid. It is not any real model.
    big = ModelConfig(vocab=32000, d_model=4096, n_layers=32, n_heads=32, d_ff=11008, max_pos=4096)
    budget = 8 * 2**30
    for itemsize, label in ((2, "float16"), (4, "float32")):
        n = max_sequences(big, budget, 4096, itemsize)
        print(f"hypothetical {big.n_layers}-layer, {big.n_heads}-head, head_dim {big.head_dim} model, "
              f"{label}: {kv_cache_bytes(big, 4096, 1, itemsize) / 2**30:.2f} GiB per 4096-token "
              f"sequence, so {n} sequences fit in 8 GiB of cache (weights not included)")


if __name__ == "__main__":
    _try("memory", _show_memory, needs=("exercise 4", "exercise 6"))

## 9. One honest wall-clock measurement

Everything so far was counted, so it is the same on every machine. The next cell is the one place
this lesson reads a clock. From one short prompt it generates continuations of several different
lengths with each decoder, several times each, and keeps the fastest run (the fastest is the least
disturbed by whatever else your machine was doing).

**These times and the speed-up ratio belong to your machine.** They move with the CPU, its
libraries and what else is running, and this lesson never checks them. What does not move is the
column of exact operation ratios beside them: the arithmetic saved is a property of the two
algorithms, and the clock ratio is that saving as delivered by one particular numpy on one
particular chip, with a fixed Python overhead per step on top.

In [ ]:
def _best_of(fn: Callable[[], Any], repeats: int = 3) -> float:
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


def _show_wall_clock() -> None:
    prompt = _make_prompt(8, SEED + 9)
    print(f"{'new tokens':>10}  {'uncached':>10}  {'cached':>10}  {'speed-up':>9}   exact op ratio")
    for n in (32, 64, 128, 256):
        slow = _best_of(lambda: generate_uncached(PARAMS, CFG, prompt, n))
        fast = _best_of(lambda: generate_cached(PARAMS, CFG, prompt, n))
        exact = generation_macs(CFG, len(prompt), n, False) / generation_macs(CFG, len(prompt), n, True)
        print(f"{n:>10}  {slow:>8.3f} s  {fast:>8.3f} s  {slow / fast:>7.1f}x   {exact:>6.1f} times")
    print("timings, and the speed-up column, are measurements of this machine; the last column is not")


if __name__ == "__main__":
    _try("wall clock", _show_wall_clock, needs=("exercise 4", "exercise 5"))

## 10. Static batching, and Exercise 7 — `make_batch_inputs`

A server handles many requests at once because one step multiplies every weight matrix once
whatever the batch size: the rows of a batch share the weights, so a step over eight rows
produces eight tokens for one pass over the parameters. But prompts have
different lengths and an array must be rectangular, so shorter prompts are **padded**. Three
things must then be true, and each is a classic bug:

1. **Pad on the left**, so every row's last position is a real token and the logits you read
   are the ones you want (Hugging Face's documentation gave the same advice as of 2026-09-29,
   see `claims.yaml`).
2. **Mask the pads**: no real token may attend to a padding slot. Without the mask every query
   also reads keys and values computed from the pad token.
3. **Position ids count real tokens only**: a request left-padded by 6 slots must still see its
   first real token at position 0, or the model reads it as if the prompt began later.

`make_batch_inputs(prompts, pad_id)` builds the three arrays `forward_full` needs: `tokens`
`(B, T)` (int, left-padded with `pad_id`), `positions` `(B, T)` (int; a real token's position
counts from 0 within its own prompt; a pad slot may be 0) and `key_mask` `(B, T)` (bool, True
for real tokens). Every prompt has at least one token.

<details><summary>💡 Hint 1 — what to think about</summary>

`T` is the longest prompt. For a prompt of length `n`, which columns are real? Which of the
three arrays share that same set of columns, and what is different about `positions` inside it?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Start from three full-size arrays: tokens filled with `pad_id`, positions all zero, mask all
`False`. For each prompt, the last `n` columns of its row get the prompt, the numbers `0..n-1`
and `True`. Nothing is written into the pad columns.
</details>

In [ ]:
def make_batch_inputs(prompts: Sequence[Sequence[int]], pad_id: int):
    """Left-pad `prompts` into (tokens, positions, key_mask), each of shape (B, T).

    Worked example: prompts [[5, 6, 7], [8]] with pad_id 0 give
        tokens    [[5, 6, 7], [0, 0, 8]]
        positions [[0, 1, 2], [0, 0, 0]]     (the real token of the short row is at position 0)
        key_mask  [[T, T, T], [F, F, T]]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_make_batch_inputs() -> None:
    tokens, positions, key_mask = make_batch_inputs([[5, 6, 7], [8], [9, 10]], 0)
    assert tokens.shape == positions.shape == key_mask.shape == (3, 3), (
        f"shapes {tokens.shape}, {positions.shape}, {key_mask.shape}: all three are (batch, longest prompt)")
    assert np.array_equal(tokens, [[5, 6, 7], [0, 0, 8], [0, 9, 10]]), (
        f"tokens are\n{tokens}\nbut padding goes on the LEFT, so the last column holds every row's "
        "last real token")
    assert key_mask.dtype == bool and np.array_equal(key_mask, [[1, 1, 1], [0, 0, 1], [0, 1, 1]]), (
        f"key_mask is\n{key_mask}\nit must be True exactly on real tokens, and boolean")
    assert np.array_equal(positions[key_mask], [0, 1, 2, 0, 0, 1]), (
        f"real-token positions are {positions[key_mask].tolist()}, expected [0, 1, 2, 0, 0, 1]: each "
        "prompt's first real token is at position 0, whatever the padding")
    fill = make_batch_inputs([[1, 2], [3]], 63)[0]
    assert fill[1, 0] == 63, "the pad slots must hold pad_id"
    assert np.issubdtype(tokens.dtype, np.integer) and np.issubdtype(positions.dtype, np.integer), (
        f"tokens are {tokens.dtype} and positions {positions.dtype}: both are ids used to index "
        "arrays, so they must be integers")
    print("make_batch_inputs: left padding, a real-tokens-only mask and per-prompt positions")


if __name__ == "__main__":
    _try("exercise 7", _check_make_batch_inputs)

In [ ]:
def generate_static_batch(params: dict, cfg: ModelConfig, prompts, n_new: int, pad_id: int = 0,
                          *, use_mask: bool = True, use_positions: bool = True):
    """Serve `prompts` together as one left-padded batch, with the cache. Given to you.

    Returns (tokens, first_logits): the (B, n_new) greedy tokens and the (B, vocab) logits of each
    row's first token. `use_mask=False` and `use_positions=False` switch off the two protections of
    exercise 7, so the next cells can measure what each one is worth: they exist to show the bugs.
    """
    tokens, positions, key_mask = make_batch_inputs(prompts, pad_id)
    b, t = tokens.shape
    lengths = np.array([len(p) for p in prompts])
    if not use_positions:
        positions = np.broadcast_to(np.arange(t), (b, t))          # pads counted as positions
    cache = KVCache(cfg, batch=b, max_len=t + n_new)
    logits = forward_full(params, cfg, tokens, positions, key_mask if use_mask else None, cache=cache)
    first_logits = np.array(logits)
    out = np.zeros((b, n_new), dtype=np.int64)
    for step in range(n_new):
        chosen = np.argmax(logits, axis=1)
        out[:, step] = chosen
        if step < n_new - 1:
            key_mask = np.concatenate([key_mask, np.ones((b, 1), dtype=bool)], axis=1)
            where = lengths + step if use_positions else np.full(b, t + step)
            logits = decode_step(params, cfg, cache, chosen, where, key_mask if use_mask else None)
    return out, first_logits


def attend_neg_inf(q, k, v, mask):
    """`attend` with the tempting mask value: -inf. Given, and deliberately wrong for a row whose
    every key is masked."""
    scores = mm(q, np.swapaxes(k, -1, -2)) / np.sqrt(q.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    return mm(softmax(scores), v)


@contextlib.contextmanager
def using(name: str, replacement: Callable[..., Any]):
    """Temporarily swap one of this notebook's own functions for another. For the demos only."""
    original = globals()[name]
    globals()[name] = replacement
    try:
        yield
    finally:
        globals()[name] = original

### Measuring the three bugs

Below, a batch of requests of different lengths is served three ways: each one alone with the cache
(the reference, a request's own result), then together as one batch with everything right, then
as a batch with one protection removed at a time. "Identical" means every one of the requests'
greedy tokens equals what it produced alone. The last lines swap in the `-inf` mask from the hint
above and count how many requests come back as `nan`.

In [ ]:
def _show_batching_traps() -> None:
    n_new = 24
    lengths = np.random.default_rng(SEED + 20).integers(3, 17, size=8).tolist()
    prompts = [_make_prompt(n, SEED + 30 + i) for i, n in enumerate(lengths)]
    alone = [generate_cached(PARAMS, CFG, p, n_new)[0] for p in prompts]
    unpadded = [i for i, n in enumerate(lengths) if n == max(lengths)]
    print(f"prompt lengths {lengths}; requests {unpadded} need no padding at all")
    variants = (("everything right", {}), ("padding not masked", {"use_mask": False}),
                ("positions count the pads", {"use_positions": False}),
                ("neither protection", {"use_mask": False, "use_positions": False}))
    for name, flags in variants:
        batch, _ = generate_static_batch(PARAMS, CFG, prompts, n_new, **flags)
        first_wrong = []
        for row in range(8):
            differs = np.nonzero(batch[row] != np.array(alone[row]))[0]
            first_wrong.append(int(differs[0]) if len(differs) else -1)
        same = sum(1 for w in first_wrong if w == -1)
        print(f"  {name:<26} {same} of 8 requests identical to serving alone; first differing "
              f"step per request (-1 = none): {first_wrong}")
    with using("attend", attend_neg_inf), np.errstate(invalid="ignore"):
        _, first = generate_static_batch(PARAMS, CFG, prompts, n_new)
    n_nan = int(np.isnan(first).any(axis=1).sum())
    print(f"with -inf as the mask value: {n_nan} of 8 requests came back as nan. {8 - len(unpadded)} "
          "of the 8 are padded; a padding slot's query has no legal key, its nan enters the "
          "next layer's keys and values, and even a weight of exactly 0 times nan is nan")


if __name__ == "__main__":
    _try("batching traps", _show_batching_traps,
         needs=("exercise 1", "exercise 3", "exercise 4", "exercise 7"))

## 11. Exercise 8 — what padding costs

A static batch pays twice for being rectangular. The prompts are padded to the longest, and the
batch keeps every row decoding until the LONGEST request has produced all its tokens, even
after a short one is done. Count it in "positions the model processes":

- the prefill processes `B × longest_prompt` positions;
- each of the `longest_generation − 1` decode steps processes `B` positions (the first token
  comes out of the prefill and the last one is never fed back);
- a request does useful work on its own `prompt_len` prompt positions and `new_tokens − 1`
  decode positions. Everything else is waste: **prompt padding** and **decode idle**.

Write `static_batch_waste(prompt_lens, new_tokens)` returning a dict with the four integer
counts `useful`, `prompt_padding`, `decode_idle` and `total`.

<details><summary>💡 Hint 1 — what to think about</summary>

Total first: rows times the positions each row is carried through. Then split it. Do the three
pieces have to add back up to the total, and can you use that to check your arithmetic?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Useful is each request's own prompt length plus its own decode positions, summed. Padding is how
far each prompt falls short of the longest. Idle is how far each generation falls short of the
longest. The total is the batch size times the prefill width plus the decode steps.
</details>

In [ ]:
def static_batch_waste(prompt_lens: Sequence[int], new_tokens: Sequence[int]) -> dict:
    """Positions processed by one static batch, split into useful work and the two kinds of waste.

    Returns {"useful", "prompt_padding", "decode_idle", "total"}, all ints, with
    useful + prompt_padding + decode_idle == total.

    Worked example: prompt_lens [2, 4] and new_tokens [3, 1]. The prefill is 2 rows × 4 = 8
    positions; there are max(new_tokens) − 1 = 2 decode steps of 2 rows = 4; total 12. Useful:
    (2 + 2) + (4 + 0) = 8. Padding 2, idle (3−3) + (3−1) = 2, and 8 + 2 + 2 = 12.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_static_batch_waste() -> None:
    with _time_limit(10.0, "this call", _MEMORY_MIB):
        _check_static_batch_waste_inner()


def _check_static_batch_waste_inner() -> None:
    got = static_batch_waste([2, 4], [3, 1])
    assert got == {"useful": 8, "prompt_padding": 2, "decode_idle": 2, "total": 12}, (
        f"got {got}; the docstring example is useful 8, prompt_padding 2, decode_idle 2, total 12")
    even = static_batch_waste([5, 5, 5], [4, 4, 4])
    assert even["prompt_padding"] == even["decode_idle"] == 0 and even["useful"] == even["total"], (
        f"three identical requests waste nothing, but got {even}")
    one_token = static_batch_waste([3, 6], [1, 1])
    assert one_token["decode_idle"] == 0 and one_token["total"] == 12, (
        f"with one new token each there are no decode steps at all, so the total is just the "
        f"prefill (2 x 6 = 12), got {one_token}")
    print("static_batch_waste: the three parts add back up to the total in every case checked")


if __name__ == "__main__":
    _try("exercise 8", _check_static_batch_waste)

In [ ]:
def make_workload(n: int, seed: int, max_gap: int = 3):
    """`n` requests: arrival steps (non-decreasing; each gap is 0..max_gap steps), prompt lengths
    and new-token counts. Given."""
    rng = np.random.default_rng(seed)
    arrivals = np.cumsum(rng.integers(0, max_gap + 1, size=n)).tolist()
    prompt_lens = rng.integers(4, 33, size=n).tolist()
    new_tokens = rng.integers(4, 65, size=n).tolist()
    return arrivals, prompt_lens, new_tokens


def _show_waste() -> None:
    _, prompt_lens, new_tokens = make_workload(8, SEED + 40)
    w = static_batch_waste(prompt_lens, new_tokens)
    print(f"8 requests, prompt lengths {prompt_lens}, new tokens {new_tokens}")
    print(f"served as one static batch: {w['total']} positions processed, "
          f"{100 * w['useful'] / w['total']:.1f}% useful, {100 * w['prompt_padding'] / w['total']:.1f}% "
          f"prompt padding, {100 * w['decode_idle'] / w['total']:.1f}% decode idle")
    if w["decode_idle"] == w["prompt_padding"]:
        share = f"the two kinds of waste are exactly equal here, {w['decode_idle']} positions each"
    else:
        bigger = "decode idle" if w["decode_idle"] > w["prompt_padding"] else "prompt padding"
        share = f"the larger share of the waste here is {bigger}"
    print(f"{share}: a short request keeps occupying its row after it is finished, and a short "
          "prompt keeps its padded slots for the whole prefill")


if __name__ == "__main__":
    _try("padding waste", _show_waste, needs=("exercise 8",))

## 12. Exercise 9 — latency in logical steps

Wall-clock latency depends on the machine. **Logical steps** do not: count one step per forward
pass of the batch, so a batch starting at step `s` produces its first tokens at `s + 1` (the
prefill) and its `j`-th token at `s + j`. The metrics are the usual ones (vLLM's documentation
defined them the same way as of 2026-09-29, see `claims.yaml`), counted in steps: **time to first token** is
`first_token − arrival`, **inter-token latency** the steps between consecutive tokens, and
**throughput** is generated tokens per step of the whole run.

The policy is static batching: requests, in arrival order, are cut into consecutive groups of
`batch_size`. A group starts once the previous group has finished AND its last member has
arrived, and runs until its longest request is done. Write `static_schedule`, returning one
`RequestTimes(batch_start, first_token, finish, batch_end)` per request, in input order:
`finish` is when THIS request's last token comes out; `batch_end` is when its batch is over.

<details><summary>💡 Hint 1 — what to think about</summary>

When can a group start? What is the group's length in steps, and which request decides it? Once
a group ends, what has to be carried over to the next one?
</details>

<details><summary>💡 Hint 2 — the approach in words</summary>

Walk the requests in groups. Keep "the step at which the server becomes free" (0 at the start).
A group starts at the later of that and its latest arrival, ends after as many steps as its
largest `new_tokens`, and every member's first token is one step after the start and its finish
is its own `new_tokens` after the start.
</details>

In [ ]:
class RequestTimes(NamedTuple):
    batch_start: int
    first_token: int
    finish: int
    batch_end: int


def static_schedule(arrivals: Sequence[int], new_tokens: Sequence[int], batch_size: int) -> list:
    """When each request's batch starts, its first token appears, it finishes, and its batch ends.

    `arrivals` is non-decreasing; every `new_tokens` entry is >= 1. Returns a list of
    `RequestTimes`, one per request, in input order.

    Worked example: arrivals [0, 2, 3], new_tokens [4, 2, 5], batch_size 2. The first group
    is requests 0 and 1: it starts at step max(0, 2) = 2 and runs 4 steps, so it ends at 6 and the
    times are (2, 3, 6, 6) and (2, 3, 4, 6). The second group is request 2 alone: it starts at
    max(6, 3) = 6, ends at 11, and its times are (6, 7, 11, 11).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_static_schedule() -> None:
    with _time_limit(10.0, "this call", _MEMORY_MIB):
        _check_static_schedule_inner()


class _Reads(list):
    """A list that counts how often it is indexed, so that a loop that never advances fails in a
    fraction of a second instead of filling memory. Given; used only by the check below."""
    def __getitem__(self, i):
        self.reads = getattr(self, "reads", 0) + 1
        if self.reads > 100_000:
            raise AssertionError("static_schedule read its inputs more than 100,000 times on a "
                                 "three-request workload: a loop that never advances to the next group")
        return super().__getitem__(i)


def _check_static_schedule_inner() -> None:
    got = static_schedule(_Reads([0, 2, 3]), _Reads([4, 2, 5]), 2)
    want = [RequestTimes(2, 3, 6, 6), RequestTimes(2, 3, 4, 6), RequestTimes(6, 7, 11, 11)]
    assert list(map(tuple, got)) == list(map(tuple, want)), (
        f"got {[tuple(t) for t in got]}, expected {[tuple(t) for t in want]}: a group starts at the "
        "later of 'server free' and 'last member arrived', and lasts as long as its longest request")
    serial = static_schedule([0, 0, 10], [3, 3, 3], 1)
    assert [t.batch_start for t in serial] == [0, 3, 10], (
        f"with batch_size 1 the starts should be [0, 3, 10] (a request may wait for the server, or "
        f"the server for a request), got {[t.batch_start for t in serial]}")
    mixed = static_schedule([0, 0], [2, 6], 2)
    assert [(t.finish, t.batch_end) for t in mixed] == [(2, 6), (6, 6)], (
        f"got {[(t.finish, t.batch_end) for t in mixed]}, expected [(2, 6), (6, 6)]: a group lasts as "
        "long as its LONGEST request, wherever in the group that one is")
    print("static_schedule: queueing behind a running batch, waiting for the last arrival, partial groups")


if __name__ == "__main__":
    _try("exercise 9", _check_static_schedule)

In [ ]:
def summarise_latency(arrivals, new_tokens, times) -> dict:
    """Logical-step metrics of a finished schedule. Given to you."""
    ttft = [t.first_token - a for t, a in zip(times, arrivals)]
    e2e = [t.finish - a for t, a in zip(times, arrivals)]
    itl = [(t.finish - t.first_token) / (n - 1) for t, n in zip(times, new_tokens) if n > 1]
    makespan = max(t.batch_end for t in times)
    return {"mean_ttft": sum(ttft) / len(ttft), "worst_ttft": max(ttft),
            "mean_e2e": sum(e2e) / len(e2e), "makespan": makespan,
            "tokens_per_step": sum(new_tokens) / makespan,
            "itl_min": min(itl), "itl_max": max(itl)}


def _sweep(load: str, arrivals, prompt_lens, new_tokens) -> dict:
    """One table: the same static policy at five batch sizes, on one workload."""
    print(f"{load}: 32 requests arriving over {arrivals[-1]} steps; {sum(new_tokens)} tokens to generate")
    print(f"{'batch':>5}  {'makespan':>8}  {'tokens/step':>11}  {'mean TTFT':>9}  {'worst TTFT':>10}  "
          f"{'inter-token':>11}  {'useful':>6}")
    summaries = {}
    for size in (1, 2, 4, 8, 16):
        times = static_schedule(arrivals, new_tokens, size)
        s = summaries[size] = summarise_latency(arrivals, new_tokens, times)
        useful = total = 0
        for lo in range(0, 32, size):
            w = static_batch_waste(prompt_lens[lo:lo + size], new_tokens[lo:lo + size])
            useful, total = useful + w["useful"], total + w["total"]
        itl = f"{s['itl_min']:.1f}-{s['itl_max']:.1f}"
        print(f"{size:>5}  {s['makespan']:>8}  {s['tokens_per_step']:>11.2f}  {s['mean_ttft']:>9.1f}  "
              f"{s['worst_ttft']:>10}  {itl:>11}  {100 * useful / total:>5.1f}%")
    return summaries


def _show_batch_size_sweep() -> None:
    busy = _sweep("busy", *make_workload(32, SEED + 50, max_gap=3))
    quiet = _sweep("quiet", *make_workload(32, SEED + 51, max_gap=100))
    for name, table in (("busy", busy), ("quiet", quiet)):
        one, big = table[1], table[16]
        faster = "higher" if big["tokens_per_step"] > one["tokens_per_step"] else "lower"
        wait = "lower" if big["mean_ttft"] < one["mean_ttft"] else "higher"
        print(f"{name} load: batch 16 has {faster} throughput ({big['tokens_per_step']:.2f} against "
              f"{one['tokens_per_step']:.2f} tokens per step) and {wait} mean time to first token "
              f"({big['mean_ttft']:.1f} against {one['mean_ttft']:.1f} steps) than batch 1")
    print("the same policy helps or hurts first-token latency depending on the load: a batch that "
          "must fill before it starts is cheap when requests are dense and costly when they are rare, "
          "and when they are rare throughput is set by how fast requests arrive, not by the batch "
          "size. Steps, not seconds: a step over 16 rows would cost more real time than a step over 1 row, "
          "and this table does not know that")


if __name__ == "__main__":
    _try("batch-size sweep", _show_batch_size_sweep, needs=("exercise 8", "exercise 9"))

## 13. Common mistakes

- **Attending before appending.** In a decode step the new key and value must be in the cache
  before the query reads it, or a token cannot attend to itself and every logit is a little wrong.
- **A position id that is the cache length.** In a padded batch a row's cache holds pad slots, so
  its next position is its own length plus the steps taken, not the number of slots in the cache.
- **Masking with `-inf`.** It works until a row has nothing legal to attend to, then it is `nan`
  and the `nan` spreads through everything downstream (section 10 measured it).
- **Padding on the right, or not masking.** The last position of a right-padded row is a pad, and
  an unmasked pad is read like any other token.
- **Trusting a speed-up you did not count.** A decoder that recomputes the prefix and returns
  the right tokens looks fine to a token check; the multiply-add count is what tells it from a cache.
- **Confusing steps with seconds.** Logical steps are exact and portable; seconds are neither.
  Say which one a number is.

In [ ]:
# The two floating-point facts behind the "-inf" mistake, computed rather than asserted.
with np.errstate(invalid="ignore"):
    print("(-inf) - (-inf) =", np.float64(-np.inf) - np.float64(-np.inf),
          "; 0.0 * nan =", np.float64(0.0) * np.float64(np.nan))

## 14. Self-check

1. Cached and uncached greedy decoding give the same tokens, and logits equal up to rounding.
   Why is that expected, rather than lucky?
   - (a) The cache stores the final logits, so nothing is recomputed
   - (b) A position's keys and values depend only on the tokens up to it (the causal mask), so
     the stored ones are what a recomputation would produce, up to floating-point rounding
   - (c) Greedy decoding always picks the same token whatever the model computes
   - (d) The cache rounds the values so both decoders agree

2. A model is deployed with twice as many layers and twice as many heads, at the same head
   width. Its KV cache per token is now:
   - (a) twice as large
   - (b) the same, because the head width is unchanged
   - (c) larger by the square root of two
   - (d) four times as large

3. A left-padded batch is served with correct position ids but no padding mask. What goes wrong?
   - (a) Nothing, because position ids already tell the model which tokens are padding
   - (b) Only the longest prompt is affected
   - (c) Every query can attend to the pad slots' keys and values, so shorter prompts change output
   - (d) The batch runs out of memory

4. In this lesson's static batching, a request arrives one step after a batch of eight has
   started. Its time to first token includes:
   - (a) the wait for the batch ahead of it to end, and for its own batch to fill
   - (b) nothing beyond the prefill of its own prompt, because it joins the running batch
   - (c) zero, because logical steps ignore queueing
   - (d) only the steps needed to generate its own tokens

5. Section 9 printed a wall-clock speed-up column next to an exact operation ratio column. Why is
   the operation ratio the same on every machine while the speed-up is not?
   - (a) Both numbers are properties of the algorithm alone
   - (b) The speed-up is exact and the operation ratio is measured
   - (c) The operation ratio grows with length because each uncached step redoes a longer prefix;
     the clock speed-up follows it, damped and distorted by per-step overhead and hardware that
     only this machine has
   - (d) The speed-up depends on the seed

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
print("self-check: work the questions above, then compare with the worked solution")

## 15. What this lesson does not cover

It serves a model with random weights, so it says nothing about quality. It keeps one contiguous
cache slab per batch, wasting the slots a short request never fills: the next lesson replaces
that with a paged cache. It batches statically: a request that finishes early holds its row, and
a new request waits for the whole batch, which is exactly what Orca's iteration-level scheduling
was built to fix (Yu et al., see `claims.yaml`). It never splits a long prefill or shares a common
prefix between requests.

## What you built

A reference decoder and a cached one that agree within a stated tolerance on every logit, the
exact arithmetic and memory accounting of both, and a static batcher whose two protections you
switched off to see what they were worth. The habit to keep: count first, then time, and always
check the fast path against the slow one on the same inputs.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (
                ("exercise 1", _check_attend, ()),
                ("exercise 2", _check_generate_uncached, ("exercise 1",)),
                ("exercise 3", _check_cache, ()),
                ("exercise 4", _check_decode_step, ("exercise 1", "exercise 2", "exercise 3")),
                ("exercise 5", _check_macs, ("exercise 1", "exercise 2", "exercise 3", "exercise 4")),
                ("exercise 6", _check_memory, ("exercise 3",)),
                ("exercise 7", _check_make_batch_inputs, ()),
                ("exercise 8", _check_static_batch_waste, ()),
                ("exercise 9", _check_static_schedule, ())):
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.